In [61]:
import os
from docling.document_converter import DocumentConverter
from hierarchical.postprocessor import ResultPostprocessor




In [62]:
base_dir = r"C:\Users\kilar\OneDrive\Desktop\Code Basics\Projects\Assignment - 2\Coding\Data"
file_path = r"billing\billing_codes.pdf"
source = os.path.join(base_dir, file_path)

In [63]:
def load_document(source : str):
    """
    Load a document from the specified source path.

    Args:
        source (str): The path to the document file.
    """
    #create an instance of DocumentConverter
    converter = DocumentConverter()

    # Load the document using the converter
    result = converter.convert(source)

    # Postprocessing the document
    ResultPostprocessor(result).process()

    return result

doc = load_document(source)





Loading weights: 100%|██████████| 770/770 [00:00<00:00, 1423.06it/s]
Path or String-sources must point to a local path that exists or to HTTP or HTTPS URLs. Got: billing_codes.pdf


In [64]:
result = doc.document # this is the Docling Document

In [65]:
markdown_doc = result.export_to_markdown() # Docling will convert Docling document into a Mark down Document

print(markdown_doc[:1000])

## Insurance Billing Code Reference

ICD-10 Diagnosis Codes, Procedure Codes &amp; Insurer Package Rates

MediAssist Health Network Central Billing Office Document ref: BILL-CODE-010 · Version 8.0 Access: Billing Executives &amp; Admin

## Introduction

This reference maps the diagnosis and procedure codes used in MediAssist claim submissions to insurer package rates. Accurate coding is the single biggest determinant of timely claim approval.

- ICD-10 codes describe the diagnosis (why the patient was treated).

- Procedure codes describe the intervention (what was done).

- Specificity matters: an unspecified code invites a query or rejection. Always code to the highest level of detail supported by the clinical record.

## Note

Package rates below are indicative MediAssist negotiated rates. Final settlement depends on the patient's policy, sum insured, co-pay and sub-limits.

## 1. Top 30 Diagnosis Codes Used at MediAssist

| ICD-10   | Description                              | Typi

In [66]:
print(type(result))
print(len(result.texts), "text items")
print(len(result.tables), "tables")

<class 'docling_core.types.doc.document.DoclingDocument'>
59 text items
7 tables


In [67]:
from docling.chunking import HybridChunker
from transformers import AutoTokenizer




In [68]:
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "sample_collection"

In [69]:
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)

chunker = HybridChunker(tokenizer = tokenizer, max_tokens = 512, merge_peers = True)

chunks = list(chunker.chunk(result))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1442 > 512). Running this sequence through the model will result in indexing errors


In [70]:
from langchain_core.documents import Document
pdf_docs = []
for chunk in chunks:
    pdf_docs.append(Document(
                    page_content = chunker.contextualize(chunk=chunk),
                    metadata = {
                        "source" : source,
                        "text": chunk.text,
                        "headings": chunk.meta.headings or [],
                        "page": chunk.meta.doc_items[0].prov[0].page_no
                        
                    }
    ))
print(f"Created {len(pdf_docs)} chunks")
if len(pdf_docs) > 0:
    print(f"\nSample chunk (note the heading prepended by serialize()):")
    print(pdf_docs[0].page_content[:100])
    print(pdf_docs[0].metadata.get("headings"))
else:
    print("\nNo chunks were created. Please check if chunk_iter was already consumed or if the PDF was processed correctly.")

Created 16 chunks

Sample chunk (note the heading prepended by serialize()):
Insurance Billing Code Reference
ICD-10 Diagnosis Codes, Procedure Codes & Insurer Package Rates
Med
['Insurance Billing Code Reference']


In [72]:
from qdrant_client import QdrantClient, models

client = QdrantClient(path = "./qdrant2")



In [73]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import FastEmbedSparse

Dense_Embeddings = HuggingFaceEmbeddings(model_name = EMBED_MODEL) # Dense embeddings

Sparse_Embeddings = FastEmbedSparse(model_name = "Qdrant/bm25") # Sparse embeddings


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3135.96it/s]


In [74]:
dense_vecs = Dense_Embeddings.embed_documents([doc.page_content for doc in pdf_docs])
sparse_vecs = Sparse_Embeddings.embed_documents([doc.page_content for doc in pdf_docs])

In [75]:
from qdrant_client.models import Distance, VectorParams

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config = {
        "dense" : VectorParams(size = len(dense_vecs[0]), 
                               distance = Distance.COSINE),
        
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(modifier=models.Modifier.IDF)
    },
    
)

True

In [76]:
import uuid

points = []
for i, doc in enumerate(pdf_docs):
    point = models.PointStruct(
        id = str(uuid.uuid4()),
        vector = {
            "dense" : dense_vecs[i],
            "sparse" : models.SparseVector(
                    indices=sparse_vecs[i].indices,
                    values=sparse_vecs[i].values,)
        
        },
        payload = {
            "source" : doc.metadata.get("source"),
            "text" : doc.metadata.get("text"),
            "headings" : doc.metadata.get("headings"),
            "page" : doc.metadata.get("page")
        }
    )

    points.append(point)



In [77]:
client.upsert(collection_name=COLLECTION_NAME, points=points,wait = True)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [78]:
print(client.count(COLLECTION_NAME))

count=16


Retreiver

In [79]:
query = "What are the billing codes for medical procedures?"

Dense_query = Dense_Embeddings.embed_query(query)
sparse_query = Sparse_Embeddings.embed_query(query)




RRF merges your dense and sparse lists using rank position only. It never looks at the text. Cheap, runs inside Qdrant, gives you a sensible broad candidate set.

- instead of looking on what bases they are ranked, we just care about there ranking

Exactly right. The position is treated as the signal; the score that produced it is discarded.

Worth being precise about one thing, though — RRF discards the scores but not the basis. Each method still ranked by its own criterion: dense sorted by semantic similarity, BM25 sorted by keyword overlap. That work is baked into the ordering. RRF just declines to compare the raw numbers across methods, because they’re incommensurable.

So it’s less “we don’t care why they ranked things” and more “we trust each method’s ordering within its own list, and refuse to compare their scoring scales.”

The cost of that choice: RRF can’t tell a confident match from a marginal one. If dense search returns a chunk at 0.94 similarity and another at 0.31, RRF sees only “1st” and “2nd” — the gap vanishes. A weak top result gets the same credit as a strong one.

Which is the reason the reranker exists. The cross-encoder goes back to the actual text and asks “does this chunk really answer this question?” — recovering exactly the judgment RRF threw away, but on a small enough candidate set to be affordable.

That’s the shape of the whole pipeline: hybrid casts wide and cheap, RRF merges without needing comparable units, the reranker applies expensive judgment to the survivors. Each stage covers the previous one’s blind spot.

In [80]:
results = client.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=[
        models.Prefetch(query=Dense_query, using="dense", limit=20),
        models.Prefetch(
            query=models.SparseVector(indices=sparse_query.indices, values=sparse_query.values),
            using="sparse",
            limit=10,
        ),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF), # RRF : Reciprocal Rank Fusion
    limit=5,
    with_payload=True,
).points

In [81]:
print(results[0].payload.keys())

dict_keys(['source', 'text', 'headings', 'page'])


In [82]:
for i, r in enumerate(results, 1):
    print(i, round(r.score, 4), "|", r.payload.get("headings"), "|", r.payload["text"][:100])

1 1.0 | ['Insurance Billing Code Reference'] | ICD-10 Diagnosis Codes, Procedure Codes & Insurer Package Rates
MediAssist Health Network Central Bi
2 0.5 | ['Introduction'] | This reference maps the diagnosis and procedure codes used in MediAssist claim submissions to insure
3 0.4762 | ['7. Pre-authorisation Document Checklist', 'See also'] | Cross-reference: the operational steps for submitting, enhancing and escalating claims are detailed 
4 0.4333 | ['7. Pre-authorisation Document Checklist'] | - Completed insurer pre-authorisation form
- Admission note with provisional diagnosis and ICD-10 co
5 0.3667 | ['2. Common Procedure Codes'] |  + IOL). PROC-OPTH-01, Package (₹) = ₹26,000. PROC-OPTH-01, Setting = Day care. PROC-GAST-01, Proced


In [ ]:
from typing import List

from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from qdrant_client import models


class HybridRetriever(BaseRetriever):
    client: object
    collection_name: str
    dense_embeddings: object
    sparse_embeddings: object
    k: int = 10

    class Config:
        arbitrary_types_allowed = True

    def _get_relevant_documents(self, query: str, **kwargs) -> List[Document]:
        dq = self.dense_embeddings.embed_query(query)
        sq = self.sparse_embeddings.embed_query(query)

        hits = self.client.query_points(
            collection_name=self.collection_name,
            prefetch=[
                models.Prefetch(query=dq, using="dense", limit=20),
                models.Prefetch(
                    query=models.SparseVector(
                        indices=sq.indices,
                        values=sq.values,
                    ),
                    using="sparse",
                    limit=20,
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=self.k,
            with_payload=True,
        ).points

        return [
            Document(
                page_content=h.payload["text"],
                metadata={k: v for k, v in h.payload.items() if k != "text"},
            )
            for h in hits
        ]

C:\Users\kilar\AppData\Local\Temp\ipykernel_25796\3809108743.py:8: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  class HybridRetriever(BaseRetriever):


```
dense 20 ─┐
          ├─ RRF ─→ 10 candidates ─→ reranker ─→ 3 ─→ LLM
sparse 20 ┘
```

In [ ]:
hybrid_retriever = HybridRetriever(
    client=client,
    collection_name=COLLECTION_NAME,
    dense_embeddings=Dense_Embeddings,
    sparse_embeddings=Sparse_Embeddings,
    k=10,
) # Coz langchain retriever expects a retriever object, not a list of results.

In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder


In [ ]:
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

cross_encoder = HuggingFaceCrossEncoder(model_name = CROSS_ENCODER_MODEL)

c:\Users\kilar\OneDrive\Desktop\Code Basics\Projects\Assignment - 2\Coding\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kilar\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\kilar\OneDrive\Desktop\Code Basics\Project

In [ ]:


# Wrap it as a LangChain document compressor
reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=3           # keep only top-3 after reranking
)

In [ ]:

reranking_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever= hybrid_retriever
)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

In [ ]:
import requests
r = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.getenv('GROQ_API_KEY')}"},
)
for m in r.json()["data"]:
    print(m["id"])

meta-llama/llama-prompt-guard-2-86m
whisper-large-v3-turbo
openai/gpt-oss-120b
openai/gpt-oss-20b
canopylabs/orpheus-v1-english
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-22m
allam-2-7b
canopylabs/orpheus-arabic-saudi
qwen/qwen3.8-27b
whisper-large-v3


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-20b",
    temperature = 0,
    max_tokens = None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)



In [ ]:
from langchain_core.prompts import ChatPromptTemplate
system_prompt = """You are a helpful Medibot assistant.
Answer the customer's question using ONLY the information provided in the context below.
If the answer is not in the context, say "I don't have that information."
Keep answers concise and friendly.

Context:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [ ]:
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
reranking_rag_chain = create_retrieval_chain(
    reranking_retriever,
    create_stuff_documents_chain(llm, prompt)
)


In [ ]:
query = "What are the billing codes for medical procedures?"
response = reranking_rag_chain.invoke({"input": query})

In [ ]:

print("=== Answer ===")
print(response["answer"])

print("\n=== Context Documents ===")
for i, doc in enumerate(response["context"], 1):
    print(f"\n[Document {i}]")
    print(doc.metadata.get("headings"), "|", doc.metadata.get("source"))
    print(doc.page_content[:300])

=== Answer ===
I don't have that information.

=== Context Documents ===

[Document 1]
['Insurance Billing Code Reference'] | C:\Users\kilar\OneDrive\Desktop\Code Basics\Projects\Assignment - 2\Coding\Data\billing\billing_codes.pdf
ICD-10 Diagnosis Codes, Procedure Codes & Insurer Package Rates
MediAssist Health Network Central Billing Office Document ref: BILL-CODE-010 · Version 8.0 Access: Billing Executives & Admin

[Document 2]
['Introduction'] | C:\Users\kilar\OneDrive\Desktop\Code Basics\Projects\Assignment - 2\Coding\Data\billing\billing_codes.pdf
This reference maps the diagnosis and procedure codes used in MediAssist claim submissions to insurer package rates. Accurate coding is the single biggest determinant of timely claim approval.
- ICD-10 codes describe the diagnosis (why the patient was treated).
- Procedure codes describe the interve

[Document 3]
['7. Pre-authorisation Document Checklist', 'See also'] | C:\Users\kilar\OneDrive\Desktop\Code Basics\Projects\Assignment - 